
# Typed GP with Depth Constraints and ASCII Tree Visualization (DEAP)

This notebook shows three things with **DEAP** genetic programming:

1. **Constrain depth** to control bloat (hard limits + soft parsimony).
2. **Typed GP** mixing boolean/numeric subtrees via `if_then_else`.
3. A small **ASCII tree** visualizer for interpretability.

> Tip: If DEAP isn't installed in your environment, run the first cell to install it.


In [ ]:
import numpy as np
import operator, copy
from functools import partial

from deap import base, creator, tools, gp

# ---- typed GP aliases ----
Num  = float   # label for numeric subtrees
Bool = bool    # label for boolean subtrees

# ---- ASCII tree utilities ----
def _to_nested(expr):
    """Turn a DEAP gp.PrimitiveTree into nested tuples: (name, [children...])."""
    stack = []
    for node in reversed(expr):  # prefix order
        if getattr(node, "arity", 0) == 0:
            stack.append((str(node), []))
        else:
            kids = [stack.pop() for _ in range(node.arity)][::-1]
            stack.append((str(node), kids))
    return stack[0] if stack else ("<empty>", [])

def print_ascii_tree(expr):
    """Pretty-print a tree with box-drawing characters."""
    root = _to_nested(expr)
    def _rec(node, prefix="", is_last=True):
        name, children = node
        connector = "└─ " if is_last else "├─ "
        print(prefix + connector + name)
        new_prefix = prefix + ("   " if is_last else "│  ")
        for i, ch in enumerate(children):
            _rec(ch, new_prefix, i == len(children)-1)
    name, children = root
    print(name)
    for i, ch in enumerate(children):
        _rec(ch, "", i == len(children)-1)


## Build a typed primitive set with boolean subtrees and `if_then_else`

In [ ]:
# Number of numeric inputs (x0..x{n_inputs-1})
n_inputs = 5

# Typed primitive set: inputs are Num, output Num
pset = gp.PrimitiveSetTyped("MAIN_T", [Num] * n_inputs, Num)
for i in range(n_inputs):
    pset.renameArguments(**{f"ARG{i}": f"x{i}"})

# Numeric primitives (vectorized)
def div_safe(a, b, eps=1e-8):
    return a / np.where(np.abs(b) < eps, eps, b)
def log1p_safe(x, clip_min=-0.999999):
    return np.log1p(np.clip(x, clip_min, None))

pset.addPrimitive(lambda a,b: a+b, [Num, Num], Num, name="add")
pset.addPrimitive(lambda a,b: a-b, [Num, Num], Num, name="sub")
pset.addPrimitive(lambda a,b: a*b, [Num, Num], Num, name="mul")
pset.addPrimitive(div_safe,       [Num, Num], Num, name="div_safe")
pset.addPrimitive(log1p_safe,     [Num],      Num, name="log1p_safe")

# Boolean primitives
pset.addPrimitive(lambda a,b: a < b, [Num, Num], Bool, name="lt")
pset.addPrimitive(lambda a,b: a > b, [Num, Num], Bool, name="gt")

# Branching primitive: if cond then a else b
pset.addPrimitive(lambda c,a,b: np.where(c, a, b), [Bool, Num, Num], Num, name="ifte")

# Ephemeral constant (typed as Num automatically)
from functools import partial
from numpy.random import uniform
pset.addEphemeralConstant("rand", partial(uniform, -1.0, 1.0))

## Toolbox: initialization, depth constraints, evaluation

In [ ]:
# Create individual & fitness types
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if not hasattr(creator, "IndividualTyped"):
    creator.create("IndividualTyped", gp.PrimitiveTree, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Tree generators (init and mutation), with bounded depths
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=3)      # init depth
toolbox.register("individual", tools.initIterate, creator.IndividualTyped, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("compile", gp.compile, pset=pset)

# Variation operators
toolbox.register("mate", gp.cxOnePoint)
toolbox.register("expr_mut", gp.genHalfAndHalf, min_=0, max_=5, pset=pset)  # mutation growth cap
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr_mut, pset=pset)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("clone", copy.deepcopy)

# HARD LIMITS on tree height (bloat control). You can swap to key=len for total node cap.
toolbox.decorate("mate",   gp.staticLimit(key=operator.attrgetter("height"), max_value=6))
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=6))

# Data for demonstration
rng = np.random.RandomState(123)
X = rng.randn(400, n_inputs)
y = (2.0*X[:,0] - 0.5*X[:,1] + np.tanh(X[:,2]) + (X[:,3] > X[:,4]).astype(float) + rng.randn(400)*0.1)

# Fitness: absolute Pearson corr with y minus parsimony (soft bloat control)
def evaluate(individual):
    func = toolbox.compile(individual)
    cols = [X[:, i] for i in range(X.shape[1])]
    with np.errstate(divide="ignore", invalid="ignore", over="ignore", under="ignore"):
        f = func(*cols)
    f = np.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0)
    f_c = f - f.mean(); y_c = y - y.mean()
    num = float(np.dot(f_c, y_c))
    den = float(np.linalg.norm(f_c) * np.linalg.norm(y_c) + 1e-12)
    corr = 0.0 if den == 0 else abs(num/den)
    penalty = 0.001 * len(individual)   # softer bias toward small trees
    return (corr - penalty,)

toolbox.register("evaluate", evaluate)

## Evolve with tournaments and print ASCII best tree each generation

In [ ]:
pop_size   = 60
generations = 10
elite       = 1
cx_prob     = 0.6
mut_prob    = 0.3
rng = np.random.RandomState(42)

pop = toolbox.population(n=pop_size)
hof = tools.HallOfFame(maxsize=1)

# Evaluate initial population
invalid = [ind for ind in pop if not ind.fitness.valid]
for ind in invalid:
    ind.fitness.values = toolbox.evaluate(ind)
hof.update(pop)

for gen in range(generations):
    # Selection with elites protected
    offspring = tools.selBest(pop, elite) + toolbox.select(pop, len(pop) - elite)
    offspring = list(map(toolbox.clone, offspring))

    # Randomized pairings among non-elites
    start = elite
    idx = np.arange(start, len(offspring))
    rng.shuffle(idx)
    for a, b in zip(idx[::2], idx[1::2]):
        if rng.rand() < cx_prob:
            gp.cxOnePoint(offspring[a], offspring[b])
            del offspring[a].fitness.values
            del offspring[b].fitness.values

    # Mutate non-elites
    for i in range(start, len(offspring)):
        if rng.rand() < mut_prob:
            toolbox.mutate(offspring[i])
            del offspring[i].fitness.values

    # Evaluate new/changed
    invalid = [ind for ind in offspring if not ind.fitness.valid]
    for ind in invalid:
        ind.fitness.values = toolbox.evaluate(ind)

    # Update pop + HOF
    pop[:] = offspring
    hof.update(pop)

    # Log best
    best = hof[0]
    print(f"[gen {gen:02d}] best fitness = {best.fitness.values[0]:.6f}, size={len(best)}, height={best.height}")
    print_ascii_tree(best)
    print("-"*60)

best = hof[0]
print("\nFinal best tree (LISP):\n", best)
print("\nFinal best tree (ASCII):")
print_ascii_tree(best)